In [1]:
import qim3d
from tifffile import TiffFile
from pprint import pprint
import dask.array as da

## 1. Downloading and Inspecting Available Datasets

Before diving into our analysis, we need to fetch some example data. The `qim3d.io.Downloader` is a simple interface for listing, downloading, and loading the built-in demonstration files provided with `qim3d`.  
We begin by creating a `Downloader` object. This sets up the connection to the remote archive (or local cache) where example datasets live.

In [3]:
downloader = qim3d.io.Downloader()

Calling `downloader.list_files()` prints out the names of all datasets you can fetch. This gives you a quick overview of what’s ready to download

In [4]:
downloader.list_files()


╭──────╮
│ Coal │
╰──────╯
Coal.CoalBrikett                                  (2.24GB)
Coal.CoalBrikettZoom_DOWNSAMPLED                  (238.50MB)
Coal.CoalBrikett_Zoom                             (3.73GB)

╭────────╮
│ Corals │
╰────────╯
Corals.Coral_1                                    (2.27GB)
Corals.Coral_2                                    (2.38GB)
Corals.Coral2_DOWNSAMPLED                         (152.66MB)
Corals.Coral_1_1                                  (1.83GB)
Corals.Coral_1_2                                  (1.83GB)
Corals.Coral_1_3                                  (1.83GB)
Corals.MexCoral                                   (2.24GB)

╭─────────────╮
│ Cowry_Shell │
╰─────────────╯
Cowry_Shell.Cowry_DOWNSAMPLED                     (116.91MB)
Cowry_Shell.Cowry_Shell                           (1.83GB)

╭──────╮
│ Crab │
╰──────╯
Crab.HerrmitCrab                                  (2.38GB)
Crab.OkinawaCrab                                  (1.86GB)

╭───────────────╮
│ Deer_Man

Once you know the name of the file you want, you can load it with a single call. Here we choose the Hourglass example, passing load_file=True to download and instantiate it immediately.

In [5]:
data = downloader.Hourglass.Hourglass(load_file=True)

https://archive.compute.dtu.dk/download/public/projects/viscomp_data_repository/Hourglass/Hourglass.tif
3.73GB [08:47, 7.58MB/s]                                                        

Loading Hourglass.tif
Using virtual stack


By default, the volume is loaded with `virtual_stack=True`, which means it isn’t fully read into RAM. Instead, only its metadata and index structure are loaded up front, and individual portions of the volume are then lazily loaded only when explicitly requested.

## 2. Exploring Data Properties & Basic Visualization


To explore the data, we first read the TIFF metadata. Using a simple loop, we can display all tags:

- `BitsPerSample: 32` – each pixel is a 32-bit value.  
- `ImageDescription` – ImageJ header:
  - `ImageJ=1.53a` (version)  
  - `images=1000` (frames)  
  - `slices=1000` (Z-slices)  
  - `loop=false` (no loop)  
  - `min`/`max` (default display range)  
- `ImageLength: 1000` – slice height (rows).  
- `ImageWidth: 1000` – slice width (columns).  
- `NewSubfileType: 0 (UNDEFINED)` – normal image (not thumbnail or mask).  
- `PhotometricInterpretation: 1 (MINISBLACK)` – grayscale (0 = black, higher = lighter).  
- `RowsPerStrip: 1000` – one strip per full slice.  
- `SampleFormat: 3 (IEEEFP)` – pixels stored as IEEE float32.  
- `SamplesPerPixel: 1` – single-channel grayscale.  
- `StripByteCounts: (4000000,)` – bytes per strip (1000 × 1000 × 4).  
- `StripOffsets: (240,)` – file offset (in bytes) where pixel data begins.  

From these tags we see that the volume is **1000 × 1000 × 1000**, uses **float32** pixels, and is **single-channel grayscale**.  



In [6]:
with TiffFile("Hourglass/Hourglass.tif") as tif:
    # dump the first page’s tags
    tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
pprint(tags)

{'BitsPerSample': 32,
 'ImageDescription': 'ImageJ=1.53a\n'
                     'images=1000\n'
                     'slices=1000\n'
                     'loop=false\n'
                     'min=-5.077037811279297\n'
                     'max=1768.756591796875',
 'ImageLength': 1000,
 'ImageWidth': 1000,
 'NewSubfileType': <FILETYPE.UNDEFINED: 0>,
 'PhotometricInterpretation': <PHOTOMETRIC.MINISBLACK: 1>,
 'RowsPerStrip': 1000,
 'SampleFormat': <SAMPLEFORMAT.IEEEFP: 3>,
 'SamplesPerPixel': 1,
 'StripByteCounts': (4000000,),
 'StripOffsets': (240,)}


Later it is time to try to visualize it. QIM3D offers multiple visualization methods, but for a first‐time exploration the **orthogonal slicer** is especially informative. 

In [7]:
qim3d.viz.slicer_orthogonal(data, color_map="grey")

This view shows three perpendicular cross-sections (Axial (Z), Coronal (Y), Sagittal (X)), each with its own slider to adjust the slice index in real time. It’s a fast, intuitive way to understand your 3D structure from every major plane.

## 3. Segmentation and Particle Counting

Now that we’ve explored the raw volume, let’s build a simple pipeline to:

1. **Segment** the individual particles in the hourglass.  
2. **Clean up** the binary mask with morphology.  
3. **Label** each connected component.  
4. **Count** and report the total number of particles.

In [ ]:
data_da = da.from_array(data)
data_da

dask.array<array, shape=(1000, 1000, 1000), dtype=>f4, chunksize=(322, 322, 322), chunktype=numpy.ndarray>

: 